# конспект новостей с помощью Gemini

кидаем ссылку, парсим из нее новость, сохраняем в файл, далее прогоняем через модель по ключу и получаем конспект и сохраняем его в вфайл

In [7]:
# Установка зависимостей (выполните эту ячейку один раз)
%pip install requests beautifulsoup4 pandas python-dotenv google-generativeai

I0423 16:19:53.330398 12440471 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(95, generation: 1)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
from dotenv import load_dotenv
import google.generativeai as genai

In [9]:
# ключ из .env файл
load_dotenv()

url = input("Введите ссылку на новость: ")

In [10]:
# тут парсинг
print(f"Парсинг данных по ссылке: {url}")
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')
title = soup.find('title').text.strip() if soup.find('title') else "Без заголовка"

# берем текст из разных параграфов
paragraphs = soup.find_all('p')
text_content = "\n".join([p.text.strip() for p in paragraphs if p.text.strip()])

df = pd.DataFrame({'url': [url], 'title': [title], 'content': [text_content]})
df.to_csv('input.csv', index=False, encoding='utf-8')
print("✅ Исходные данные успешно сохранены в input.csv")

Парсинг данных по ссылке: https://www.newsvl.ru/vlad/2026/04/23/237981/
✅ Исходные данные успешно сохранены в input.csv


In [ ]:
api_key = os.getenv("GEMINI_API_KEY")

genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.5-flash')

prompt = f"""Пожалуйста, сделай качественное и краткий конспект следующей новости.\n
Заголовок: {title}\n
Текст: {text_content}\n"""

print("Генерация саммаризации с помощью Gemini...")
response = model.generate_content(prompt)
summary = response.text

print("\n=== РЕЗУЛЬТАТ (САММАРИЗАЦИЯ) ===\n")
print(summary)

Генерация саммаризации с помощью Gemini...

=== РЕЗУЛЬТАТ (САММАРИЗАЦИЯ) ===

Масштабный проект планировки территории Змеинки во Владивостоке (около 55 га) предусматривает строительство новых жилых домов (высотных и среднеэтажных), рассчитанных на 5200 жителей, а также разнообразной социальной инфраструктуры: несколько детских садов, больницу/поликлинику, спортивные комплексы (включая крытый), торговые центры и гостиницы.

Ключевые аспекты проекта:
*   **Сохранение сопки Змеиной:** Планируется превратить её в рекреационную зону.
*   **Снос:** Для реализации проекта будет снесён существующий частный сектор, гаражные кооперативы и заброшенные промышленные объекты.
*   **Инфраструктура:** Предусмотрена реконструкция дорог и строительство многоуровневых паркингов.

Общественные обсуждения проекта проводятся до 29 апреля, и принять в них участие могут собственники недвижимости, проживающие в границах планируемой территории.


In [ ]:
with open('output_results.txt', 'w', encoding='utf-8') as f:
    f.write(summary)
print("\n✅ Результат успешно сохранен в output_results.txt")


✅ Результат успешно сохранен в output_results.txt
